# P1.01 · Minimal scorecard (WoE/IV + logistic regression)
**End-to-end credit decision engine** · **L1** tier

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/1thirteeng3/ds-ai-cookbook/blob/colab/en/projects/P1-credit/P1.01-e2e-credit-decision-engine/notebooks/10-minimal-scorecard.ipynb)

| Level | Colab runtime | Time (smoke / full) | Document |
|---|---|---|---|
| 🟢 L1 | cpu | 2 / 10 min | [index.en.md](../index.en.md) |

> From the Kaggle file to an approval policy in one notebook: optimal binning, WoE, logistic regression, points table and cut-off. It is the industry standard for application scoring and the reference the L3 and L4 tiers must beat.

## 0. Parameters

`RUN_MODE="smoke"` uses 10% of the data (or the synthetic twin with `OFFLINE=True`) and runs in minutes. `RUN_MODE="full"` uses the whole dataset and produces the official results. `OFFLINE=True` exists only for tests and CI: never draw conclusions from it.

In [ ]:
# Parameters (papermill overrides this cell)
RUN_MODE = "smoke"   # "smoke" | "full"
SEED = 42
DATA_DIR = "data"    # Colab + Drive: "/content/drive/MyDrive/cookbook-data"
OFFLINE = False      # True: synthetic twin (tests/CI). Never for full results.
SAMPLE_FRAC = 0.1    # smoke only
N_BOOT = 200         # bootstrap resamples for metric uncertainty
PDO, BASE_SCORE, BASE_ODDS = 20, 600, 50   # scorecard scale
APPROVAL_RATE = 0.70                        # simple L1 policy

## 1. Setup

On Colab, installs the `cookbook` package from the private repo (`GH_TOKEN` secret) and the exact versions used for the results. NumPy, pandas and scikit-learn keep Colab's versions and are recorded in the result file. If the install upgrades any of them, use *Runtime > Restart session* and run from section 0. To download Home Credit, create the `KAGGLE_API_TOKEN` secret (or `KAGGLE_USERNAME` and `KAGGLE_KEY`) and accept the competition rules on kaggle.com with the same account.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = {
    "optbinning": "1.0.0",
    "lightgbm": "4.7.0",
    "optuna": "5.0.0",
    "mlflow": "3.18.0",
    "fastapi": "0.162.0",
    "httpx": "0.28.1",
    "polars": "2.1.0",
    "pyarrow": "26.0.1",
    "kaggle": "2.2.5",
    "joblib": "1.6.0",
    "matplotlib": "3.11.2",
}

if IN_COLAB and not sys.modules.get("cookbook"):
    from google.colab import userdata

    token = userdata.get("GH_TOKEN")
    pins = " ".join(f"{k}=={v}" for k, v in PACKAGES.items())
    %pip install -q uv
    !uv pip install --system -q {pins} "cookbook[credit,kaggle] @ git+https://{token}@github.com/1thirteeng3/cookbokk.git@main"

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from cookbook import credit as cr
from cookbook import home_credit as hc
from cookbook.data import sample
from cookbook.env import colab_secrets_to_env, hardware_info, quiet_logs, set_seeds, versions
from cookbook.results import Timer, save_result

quiet_logs()  # before heavy imports: silences optional-solver probing
from optbinning import BinningProcess, OptimalBinning, Scorecard
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss

set_seeds(SEED)
colab_secrets_to_env()
if RUN_MODE == "full" and OFFLINE:
    raise ValueError("RUN_MODE='full' needs real data (OFFLINE=False).")

SOURCE = "home-credit-synthetic" if OFFLINE else "home-credit"
ARTIFACTS = Path(DATA_DIR) / "artifacts" / "P1.01" / RUN_MODE
ARTIFACTS.mkdir(parents=True, exist_ok=True)
HW = hardware_info()
VERSIONS = versions([*PACKAGES, "numpy", "pandas", "scikit-learn"])
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.width", 140)
print(SOURCE, RUN_MODE, ARTIFACTS, HW)

## 2. Data

`application_train.csv` has one row per credit application and `TARGET = 1` when the client had payment difficulties. We apply the project's minimal cleaning (`hc.clean`): the sentinel `DAYS_EMPLOYED = 365243` becomes missing and gets a flag, because the sentinel itself is informative (almost all are pensioners).

**Split:** the data has no application date, so there is no out-of-time split. We use 60/20/20 by ID *hash* (`hc.split`): the same client always lands in the same partition, in every notebook and every download. L1 trains on `train` and measures on `test`; `valid` is reserved for decisions (cut-off, calibration) in the next tiers.

In [ ]:
n_twin = int(307_511 * (SAMPLE_FRAC if RUN_MODE == "smoke" else 1))
app, bureau, DATASET = hc.load(SOURCE, DATA_DIR, n_synthetic=n_twin, seed=SEED)
if RUN_MODE == "smoke" and not OFFLINE:
    app = sample(app, SAMPLE_FRAC, SEED, stratify=hc.TARGET)
    bureau = bureau[bureau[hc.ID].isin(app[hc.ID])]
DATASET["sample_frac"] = SAMPLE_FRAC if RUN_MODE == "smoke" else 1.0
DATASET["n_rows"] = len(app)
print(app.shape, bureau.shape, f"default rate = {app[hc.TARGET].mean():.4f}")

In [ ]:
raw_app = app.copy()            # kept only for the pitfall demo in section 6
app = hc.clean(app)
app["CREDIT_INCOME"] = app["AMT_CREDIT"] / app["AMT_INCOME_TOTAL"]
app["ANNUITY_INCOME"] = app["AMT_ANNUITY"] / app["AMT_INCOME_TOTAL"]
app["CREDIT_TERM"] = app["AMT_CREDIT"] / app["AMT_ANNUITY"]
app["SPLIT"] = hc.split(app[hc.ID])

train, valid, test = (app[app["SPLIT"] == s] for s in ("train", "valid", "test"))
y_train, y_test = train[hc.TARGET].to_numpy(), test[hc.TARGET].to_numpy()
app.groupby("SPLIT")[hc.TARGET].agg(rows="size", default_rate="mean")

## 3. Solution

### 3.1 Baselines

Two mandatory floors: a constant PD (Gini 0, no ranking) and the best single variable (`EXT_SOURCE_2`, an external score). Any model must beat the second by a clear margin to justify its complexity.

In [ ]:
ext2 = -test["EXT_SOURCE_2"].fillna(train["EXT_SOURCE_2"].median())   # higher = riskier
BASELINES = {
    "constant PD": {"gini": 0.0},
    "EXT_SOURCE_2 only": {"gini": cr.gini(y_test, ext2), "ks": cr.ks(y_test, ext2)},
}
pd.DataFrame(BASELINES).T

### 3.2 Candidate variables

We pick variables a credit committee understands: external scores, age, employment length, income commitment, education, occupation. `CODE_GENDER` is **left out on purpose**: it is a protected attribute and is used only for fairness checks (section 8 of the document). Age is used, but discussed under Risks.

In [ ]:
FEATURES = [
    "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "DAYS_BIRTH", "DAYS_EMPLOYED",
    "DAYS_EMPLOYED_ANOM", "DAYS_ID_PUBLISH", "DAYS_LAST_PHONE_CHANGE", "AMT_CREDIT",
    "AMT_GOODS_PRICE", "CREDIT_INCOME", "ANNUITY_INCOME", "CREDIT_TERM",
    "REGION_RATING_CLIENT", "DEF_30_CNT_SOCIAL_CIRCLE", "NAME_EDUCATION_TYPE",
    "NAME_INCOME_TYPE", "OCCUPATION_TYPE", "FLAG_OWN_CAR",
]
CATEGORICAL = ["NAME_EDUCATION_TYPE", "NAME_INCOME_TYPE", "OCCUPATION_TYPE", "FLAG_OWN_CAR"]
assert "CODE_GENDER" not in FEATURES

### 3.3 Optimal binning, WoE and scorecard

`BinningProcess` groups each variable into bins with a monotonic risk trend (chosen automatically) and computes each bin's *Weight of Evidence*. Variables with *Information Value* below 0.02 are dropped. Logistic regression on the WoE becomes a points table: 600 points mean 50:1 (good:bad) odds and every 20 points double the odds.

In [ ]:
def make_scorecard(iv_min: float = 0.02) -> Scorecard:
    process = BinningProcess(
        FEATURES,
        categorical_variables=CATEGORICAL,
        selection_criteria={"iv": {"min": iv_min}},
    )
    return Scorecard(
        binning_process=process,
        estimator=LogisticRegression(max_iter=2000),
        scaling_method="pdo_odds",
        scaling_method_params={"pdo": PDO, "odds": BASE_ODDS, "scorecard_points": BASE_SCORE},
    )


scorecard = make_scorecard()
with Timer() as t_fit:
    scorecard.fit(train[FEATURES], y_train)

summary = scorecard.binning_process_.summary().sort_values("iv", ascending=False)
summary[["name", "dtype", "selected", "n_bins", "iv"]]

In [ ]:
points = scorecard.table(style="detailed")
points[points["Variable"] == "EXT_SOURCE_2"][["Bin", "Count (%)", "Event rate", "WoE", "Points"]]

## 4. Validation

Primary metric: **Gini** on the test set, with *bootstrap* standard deviation (`N_BOOT` resamples). Secondary: KS, Brier (calibration) and the train–test Gini gap (overfitting).

In [ ]:
with Timer() as t_pred:
    p_test = scorecard.predict_proba(test[FEATURES])[:, 1]
p_train = scorecard.predict_proba(train[FEATURES])[:, 1]
score_test = scorecard.score(test[FEATURES])

METRICS = {
    "gini": cr.bootstrap(cr.gini, y_test, p_test, n=N_BOOT, seed=SEED),
    "ks": cr.bootstrap(cr.ks, y_test, p_test, n=N_BOOT, seed=SEED),
    "brier": cr.bootstrap(brier_score_loss, y_test, p_test, n=N_BOOT, seed=SEED),
    "gini_train_test_gap": {"mean": round(cr.gini(y_train, p_train) - cr.gini(y_test, p_test), 6), "std": None, "n": 1},
    "n_features": {"mean": int(summary["selected"].sum()), "std": None, "n": 1},
}
pd.DataFrame(METRICS).T[["mean", "std"]]

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(score_test[y_test == 0], bins=40, alpha=0.6, density=True, label="good / bom")
ax.hist(score_test[y_test == 1], bins=40, alpha=0.6, density=True, label="bad / mau")
ax.set_xlabel("score"); ax.legend(); ax.set_title("Score distribution on test")
plt.show()

### 4.1 Simple policy: approve 70%

L1 approves the 70% with the lowest PD. The table shows, for each approval rate, the PD cut-off, the bad rate among approved and the share of bads avoided. L3 replaces this rule with a cut-off that maximizes expected profit.

In [ ]:
cutoffs = cr.cutoff_table(y_test, p_test)
at_policy = cutoffs.loc[(cutoffs["approval_rate"] - APPROVAL_RATE).abs().idxmin()]
METRICS["bad_rate_approved"] = {"mean": round(float(at_policy["bad_rate_approved"]), 6), "std": None, "n": 1}
METRICS["approval_rate"] = {"mean": APPROVAL_RATE, "std": None, "n": 1}
cutoffs

## 5. Variation axes

### 5.1 Minimum IV threshold

The stricter the IV filter, the fewer variables and the simpler the scorecard. The question is how much Gini is lost.

In [ ]:
axis_rows = []
for iv_min in (0.02, 0.05, 0.10):
    sc_axis = make_scorecard(iv_min).fit(train[FEATURES], y_train)
    p_axis = sc_axis.predict_proba(test[FEATURES])[:, 1]
    n_sel = int(sc_axis.binning_process_.summary()["selected"].sum())
    gini_axis = cr.bootstrap(cr.gini, y_test, p_axis, n=N_BOOT, seed=SEED)
    axis_rows.append({"iv_min": iv_min, "n_features": n_sel, "gini": gini_axis["mean"], "gini_std": gini_axis["std"]})
    save_result(
        recipe_id="P1.01", notebook="10-minimal-scorecard", stack="optbinning-scorecard",
        variant=f"iv_min={iv_min}", run_mode=RUN_MODE, hardware=HW, versions=VERSIONS,
        dataset=DATASET, params={"seed": SEED, "iv_min": iv_min},
        metrics={"gini": gini_axis, "n_features": {"mean": n_sel, "std": None, "n": 1}},
        time_s={"train": t_fit.seconds, "inference_per_1k": None},
    )
pd.DataFrame(axis_rows)

## 6. Pitfalls demonstrated

### 6.1 Fitting the binning on the whole dataset

**Symptom:** test Gini slightly better than reality. **Cause:** the bins and WoE "saw" the test set. **Fix:** fit binning (and any supervised transformation) on training data only, inside the pipeline.

In [ ]:
leaky_process = BinningProcess(FEATURES, categorical_variables=CATEGORICAL, selection_criteria={"iv": {"min": 0.02}})
leaky_process.fit(app[FEATURES], app[hc.TARGET])            # sees test labels: WRONG
leaky_lr = LogisticRegression(max_iter=2000).fit(leaky_process.transform(train[FEATURES], metric="woe"), y_train)
p_leaky = leaky_lr.predict_proba(leaky_process.transform(test[FEATURES], metric="woe"))[:, 1]
pd.Series({"correct (fit on train)": cr.gini(y_test, p_test), "leaky (fit on all data)": cr.gini(y_test, p_leaky)}, name="test gini")

### 6.2 Treating the 365243 sentinel as a number

**Symptom:** a huge `DAYS_EMPLOYED` bin with absurd positive values (1,000 years of employment). **Cause:** the sentinel is mixed with real values and distorts the monotonic trend. **Fix:** turn it into missing and keep a flag (`hc.clean`). Compare both tables.

In [ ]:
def bin_table(x, y):
    ob = OptimalBinning("DAYS_EMPLOYED", dtype="numerical", monotonic_trend="auto").fit(x.to_numpy(), y.to_numpy())
    return ob.binning_table.build()[["Bin", "Count (%)", "Event rate", "WoE"]]

raw_train = raw_app.loc[train.index]
print("RAW (sentinel as a number)")
display(bin_table(raw_train["DAYS_EMPLOYED"], raw_train[hc.TARGET]))
print("CLEAN (sentinel -> missing + flag)")
display(bin_table(train["DAYS_EMPLOYED"], train[hc.TARGET]))

## 7. Results

Writes the metrics in the `result.schema.json` format (the L1 table in the document is generated from it) and saves validation and test PDs for the policy comparison in notebook 32.

In [ ]:
save_result(
    recipe_id="P1.01", notebook="10-minimal-scorecard", stack="optbinning-scorecard",
    run_mode=RUN_MODE, hardware=HW, versions=VERSIONS, dataset=DATASET,
    params={"seed": SEED, "iv_min": 0.02, "pdo": PDO, "base_score": BASE_SCORE, "base_odds": BASE_ODDS,
            "approval_rate": APPROVAL_RATE},
    metrics=METRICS,
    time_s={"train": t_fit.seconds, "inference_per_1k": round(t_pred.seconds / len(test) * 1000, 6)},
)
save_result(
    recipe_id="P1.01", notebook="10-minimal-scorecard", stack="baseline-ext-source-2",
    run_mode=RUN_MODE, hardware=HW, versions=VERSIONS, dataset=DATASET, params={},
    metrics={"gini": cr.bootstrap(cr.gini, y_test, ext2.to_numpy(), n=N_BOOT, seed=SEED)},
    time_s={"train": 0.0, "inference_per_1k": None},
)

scored = pd.concat([valid, test])
pd.DataFrame({
    hc.ID: scored[hc.ID].to_numpy(),
    "SPLIT": scored["SPLIT"].to_numpy(),
    hc.TARGET: scored[hc.TARGET].to_numpy(),
    "AMT_CREDIT": scored["AMT_CREDIT"].to_numpy(),
    "pd_l1": scorecard.predict_proba(scored[FEATURES])[:, 1],
    "score_l1": scorecard.score(scored[FEATURES]),
}).to_parquet(ARTIFACTS / "l1_scores.parquet", index=False)
points.to_csv(ARTIFACTS / "l1_scorecard_points.csv", index=False)
sorted(p.name for p in ARTIFACTS.iterdir())

## 8. Conclusion

- **What we have:** an auditable scorecard with few variables, points per bin and an explicit approval rule.
- **When it is enough:** small portfolios, strong explainability requirements, little infrastructure.
- **Limits:** misses interactions and fine relationships; the 70% rule ignores each loan's value; no monitoring.
- **Next step:** [30 · feature pipeline](30-feature-pipeline.ipynb) starts the L3 tier.